In [ ]:
%pip install neo4j google-genai python-dotenv tqdm numpy pandas --quiet

In [1]:
import os
import sys
import pickle
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from neo4j import GraphDatabase
from tqdm import tqdm

sys.path.insert(0, str(Path("..").resolve()))

load_dotenv("../.env")

True

In [2]:
from graph.client import create_vertexai_client

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

client = create_vertexai_client()

In [ ]:
def get_embedding(text: str) -> list[float]:
    result = client.models.embed_content(
        model="text-embedding-004",
        contents=text,
    )
    return result.embeddings[0].values

In [4]:
with open("ground_truth.pkl", "rb") as f:
    ground_truth = pickle.load(f)

print(f"Loaded {len(ground_truth)} questions with ground truth")
print(f"Example: {ground_truth[0]}")

Loaded 50 questions with ground truth
Example: {'question_id': 'q_0', 'question': 'Who is the spouse of the Green performer?', 'answer': 'Miquette Giraudy', 'supporting_sentences': ['Miquette Giraudy_0', 'Green (Steve Hillage album)_0']}


In [30]:
q = filter(lambda x: x["question_id"] == "q_12", ground_truth)

In [31]:
list(q)

[{'question_id': 'q_12',
  'question': 'In which borough was Callum McManaman born?',
  'answer': 'Knowsley',
  'supporting_sentences': ['Callum McManaman_0', 'Whiston, Merseyside_0']}]

In [5]:
with driver.session() as session:
    paragraphs = session.run("MATCH (p:Paragraph) RETURN count(p) as c").single()["c"]
    entities = session.run("MATCH (e:Entity) RETURN count(e) as c").single()["c"]
    mentions = session.run("MATCH ()-[r:MENTIONS]->() RETURN count(r) as c").single()["c"]

print(f"Graph stats: {paragraphs} paragraphs, {entities} entities, {mentions} MENTIONS edges")


Graph stats: 2797 sentences, 4885 entities, 6220 MENTIONS edges


In [6]:
def retrieve_semantic_only(session, query_embedding: list[float], k: int = 10) -> list[dict]:
    results = session.run(
        """
        CALL db.index.vector.queryNodes('paragraph_embeddings', $k, $embedding)
        YIELD node, score
        RETURN node.id AS id, node.text AS text, score
        """,
        embedding=query_embedding,
        k=k,
    ).data()
    
    return [{"id": r["id"], "text": r["text"], "score": r["score"]} for r in results]

In [7]:
def retrieve_graph_enhanced(
    session,
    query_embedding: list[float],
    k: int = 10,
    candidate_pool: int = 100,
    alpha: float = 0.85,
    beta: float = 0.10,
    gamma: float = 0.05,
) -> list[dict]:
    results = session.run(
        """
        // Step 1: Get candidate sentences via vector search (all have semantic scores)
        CALL db.index.vector.queryNodes('paragraph_embeddings', $pool, $embedding)
        YIELD node AS candidate, score AS semantic

        // Step 2: Collect all entities from top-5 seeds for proximity calculation
        WITH collect({node: candidate, sem: semantic}) AS candidates
        WITH candidates, 
             [c IN candidates[0..5] | c.node] AS seeds

        // Get seed entities
        UNWIND seeds AS seed
        MATCH (seed)-[:MENTIONS]->(e:Entity)
        WITH candidates, collect(DISTINCT e.name) AS seed_entity_names

        // Step 3: For each candidate, compute proximity and importance
        UNWIND candidates AS c
        WITH c.node AS candidate, c.sem AS semantic, seed_entity_names

        // Count shared entities with seeds (proximity) - cap at 1.0
        OPTIONAL MATCH (candidate)-[:MENTIONS]->(e:Entity)
        WHERE e.name IN seed_entity_names
        WITH candidate, semantic, seed_entity_names, 
             CASE WHEN count(DISTINCT e) > 5 THEN 5 ELSE count(DISTINCT e) END AS shared_capped
        
        // Count total entities (importance) - cap at 1.0
        OPTIONAL MATCH (candidate)-[:MENTIONS]->(e2:Entity)
        WITH candidate, semantic, shared_capped,
             CASE WHEN count(DISTINCT e2) > 10 THEN 10 ELSE count(DISTINCT e2) END AS entity_capped
        
        // Normalize (both now max out at 1.0)
        WITH candidate, semantic,
             toFloat(shared_capped) / 5.0 AS proximity,
             toFloat(entity_capped) / 10.0 AS importance
        
        WITH candidate, semantic, proximity, importance,
             $alpha * semantic + $beta * proximity + $gamma * importance AS score
        
        RETURN candidate.id AS id, candidate.text AS text,
               score, semantic, proximity, importance
        ORDER BY score DESC
        LIMIT $k
        """,
        embedding=query_embedding,
        pool=candidate_pool,
        k=k,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
    ).data()
    
    return [
        {
            "id": r["id"],
            "text": r["text"],
            "score": r["score"],
            "semantic": r["semantic"],
            "proximity": r["proximity"],
            "importance": r["importance"],
        }
        for r in results
    ]


In [8]:
def retrieve_ppr_enhanced(
    session,
    query_embedding: list[float],
    k: int = 10,
    num_seeds: int = 10,
    alpha: float = 0.7,
    beta: float = 0.2,
    gamma: float = 0.1,
) -> list[dict]:
    results = session.run(
        """
        // Step 1: Get seed paragraphs via vector search
        CALL db.index.vector.queryNodes('paragraph_embeddings', $num_seeds, $embedding)
        YIELD node AS seed, score AS semantic
        
        // Collect seed info
        WITH collect({id: seed.id, sem: semantic, doc: seed.doc_title}) AS seed_info,
             collect(seed) AS seed_nodes,
             collect(DISTINCT seed.doc_title) AS seed_docs
        
        // Step 2: Get entities from seeds and calculate IEF
        UNWIND seed_nodes AS seed
        MATCH (seed)-[:MENTIONS]->(e:Entity)
        WITH seed_info, seed_docs, e
        
        // Calculate entity degree (for IEF)
        OPTIONAL MATCH (other:Paragraph)-[:MENTIONS]->(e)
        WITH seed_info, seed_docs, e.name AS ent_name, count(DISTINCT other) AS ent_degree
        
        // IEF weight: rare entities get higher weight
        WITH seed_info, seed_docs, ent_name, 
             CASE WHEN ent_degree < 5 THEN 1.0 
                  WHEN ent_degree < 20 THEN 0.7
                  WHEN ent_degree < 100 THEN 0.3
                  ELSE 0.1 END AS ief_weight
        
        // Collect weighted entities
        WITH seed_info, seed_docs, collect({name: ent_name, ief: ief_weight}) AS weighted_entities
        
        // Step 3: Find candidate paragraphs connected to seed entities
        UNWIND weighted_entities AS we
        MATCH (candidate:Paragraph)-[:MENTIONS]->(e:Entity {name: we.name})
        WITH seed_info, seed_docs, candidate, we.ief AS ief
        
        // Aggregate IEF score per candidate
        WITH seed_info, seed_docs, candidate, sum(ief) AS total_ief
        
        // Step 4: Calculate document locality bonus
        WITH seed_info, candidate, total_ief,
             CASE WHEN candidate.doc_title IN seed_docs THEN 1.0 ELSE 0.0 END AS doc_bonus
        
        // Step 5: Get semantic score if candidate is a seed
        WITH candidate, total_ief, doc_bonus,
             [s IN seed_info WHERE s.id = candidate.id | s.sem][0] AS sem_from_seeds
        
        // Normalize scores
        WITH candidate,
             coalesce(sem_from_seeds, 0.0) AS semantic,
             CASE WHEN total_ief > 5.0 THEN 1.0 ELSE total_ief / 5.0 END AS proximity,
             doc_bonus AS doc_locality
        
        // Combined score: semantic + IEF-weighted proximity + document locality
        WITH candidate, semantic, proximity, doc_locality,
             $alpha * semantic + $beta * proximity + $gamma * doc_locality AS score
        
        RETURN candidate.id AS id, candidate.text AS text,
               score, semantic, proximity, doc_locality
        ORDER BY score DESC
        LIMIT $k
        """,
        embedding=query_embedding,
        num_seeds=num_seeds,
        k=k,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
    ).data()
    
    return [
        {
            "id": r["id"],
            "text": r["text"],
            "score": r["score"],
            "semantic": r["semantic"],
            "proximity": r["proximity"],
            "doc_locality": r["doc_locality"],
        }
        for r in results
    ]

In [12]:
def precision_at_k(retrieved: list[dict], ground_truth_ids: list[str], k: int) -> float:
    top_k = retrieved[:k]
    relevant = sum(1 for r in top_k if r["id"] in ground_truth_ids)

    return relevant / k if k > 0 else 0.0


def recall_at_k(retrieved: list[dict], ground_truth_ids: list[str], k: int) -> float:
    top_k = retrieved[:k]
    relevant = sum(1 for r in top_k if r["id"] in ground_truth_ids)

    return relevant / len(ground_truth_ids) if ground_truth_ids else 0.0


def mrr(retrieved: list[dict], ground_truth_ids: list[str]) -> float:
    for i, r in enumerate(retrieved):
        if r["id"] in ground_truth_ids:
            return 1.0 / (i + 1)

    return 0.0


In [13]:
example = ground_truth[0]
query = example["question"]
gt_ids = example["supporting_paragraphs"]

print(f"Query: {query}")
print(f"Ground truth paragraphs: {gt_ids}")

query_embedding = get_embedding(query)

Query: Who is the spouse of the Green performer?
Ground truth sentences: ['Miquette Giraudy_0', 'Green (Steve Hillage album)_0']


In [17]:
with driver.session() as session:
    semantic_results = retrieve_semantic_only(session, query_embedding, k=10)
    graph_results = retrieve_graph_enhanced(session, query_embedding, k=10)
    ppr_results = retrieve_ppr_enhanced(session, query_embedding, k=10)

print("=== SEMANTIC ONLY ===")
for i, r in enumerate(semantic_results[:5]):
    marker = "✓" if r["id"] in gt_ids else " "
    print(f"{i+1}. [{marker}] {r['score']:.4f} | {r['text'][:60]}...")

print("\n=== GRAPH ENHANCED (old) ===")
for i, r in enumerate(graph_results[:5]):
    marker = "✓" if r["id"] in gt_ids else " "
    print(f"{i+1}. [{marker}] {r['score']:.4f} (sem:{r['semantic']:.2f} prox:{r['proximity']:.2f} imp:{r['importance']:.2f}) | {r['text'][:50]}...")

print("\n=== PPR + IEF + DOC LOCALITY ===")
for i, r in enumerate(ppr_results[:5]):
    marker = "✓" if r["id"] in gt_ids else " "
    print(f"{i+1}. [{marker}] {r['score']:.4f} (sem:{r['semantic']:.2f} prox:{r['proximity']:.2f} doc:{r['doc_locality']:.2f}) | {r['text'][:50]}...")

=== SEMANTIC ONLY ===
1. [ ] 0.7700 | He has appeared in television shows such as "Puppets Who Kil...
2. [ ] 0.7548 | Earlier recordings made by Green for Blue Note were released...
3. [✓] 0.7531 | Green is the fourth studio album by British progressive rock...
4. [ ] 0.7525 | They separate in 2007....
5. [ ] 0.7513 | The couple had four children....

=== GRAPH ENHANCED (old) ===
1. [ ] 0.7045 (sem:0.77 prox:0.40 imp:0.20) | He has appeared in television shows such as "Puppe...
2. [ ] 0.6915 (sem:0.75 prox:0.40 imp:0.20) | Earlier recordings made by Green for Blue Note wer...
3. [✓] 0.6901 (sem:0.75 prox:0.40 imp:0.20) | Green is the fourth studio album by British progre...
4. [ ] 0.6779 (sem:0.74 prox:0.00 imp:1.00) | Green Party of the United States Chairperson Green...
5. [ ] 0.6728 (sem:0.74 prox:0.20 imp:0.40) | However, this plan was dropped and after a US tour...

=== PPR + IEF + DOC LOCALITY ===
1. [ ] 0.8171 (sem:0.74 prox:1.00 doc:1.00) | Green Party of the United States Chai

In [18]:
NUM_EVAL = 500

semantic_metrics = {"p@5": [], "p@10": [], "r@5": [], "r@10": [], "mrr": []}
graph_metrics = {"p@5": [], "p@10": [], "r@5": [], "r@10": [], "mrr": []}
ppr_metrics = {"p@5": [], "p@10": [], "r@5": [], "r@10": [], "mrr": []}

evaluation_results = []

with driver.session() as session:
    for ex in tqdm(ground_truth[:NUM_EVAL], desc="Evaluating"):
        q_emb = get_embedding(ex["question"])
        gt_ids = ex["supporting_paragraphs"]
        
        sem_res = retrieve_semantic_only(session, q_emb, k=20)
        grp_res = retrieve_graph_enhanced(session, q_emb, k=20)
        ppr_res = retrieve_ppr_enhanced(session, q_emb, k=20)

        sem_p5 = precision_at_k(sem_res, gt_ids, 5)
        sem_p10 = precision_at_k(sem_res, gt_ids, 10)
        sem_r5 = recall_at_k(sem_res, gt_ids, 5)
        sem_r10 = recall_at_k(sem_res, gt_ids, 10)
        sem_mrr = mrr(sem_res, gt_ids)
        
        grp_p5 = precision_at_k(grp_res, gt_ids, 5)
        grp_p10 = precision_at_k(grp_res, gt_ids, 10)
        grp_r5 = recall_at_k(grp_res, gt_ids, 5)
        grp_r10 = recall_at_k(grp_res, gt_ids, 10)
        grp_mrr = mrr(grp_res, gt_ids)
        
        ppr_p5 = precision_at_k(ppr_res, gt_ids, 5)
        ppr_p10 = precision_at_k(ppr_res, gt_ids, 10)
        ppr_r5 = recall_at_k(ppr_res, gt_ids, 5)
        ppr_r10 = recall_at_k(ppr_res, gt_ids, 10)
        ppr_mrr = mrr(ppr_res, gt_ids)
        
        semantic_metrics["p@5"].append(sem_p5)
        semantic_metrics["p@10"].append(sem_p10)
        semantic_metrics["r@5"].append(sem_r5)
        semantic_metrics["r@10"].append(sem_r10)
        semantic_metrics["mrr"].append(sem_mrr)
        
        graph_metrics["p@5"].append(grp_p5)
        graph_metrics["p@10"].append(grp_p10)
        graph_metrics["r@5"].append(grp_r5)
        graph_metrics["r@10"].append(grp_r10)
        graph_metrics["mrr"].append(grp_mrr)
        
        ppr_metrics["p@5"].append(ppr_p5)
        ppr_metrics["p@10"].append(ppr_p10)
        ppr_metrics["r@5"].append(ppr_r5)
        ppr_metrics["r@10"].append(ppr_r10)
        ppr_metrics["mrr"].append(ppr_mrr)
        
        evaluation_results.append({
            "question_id": ex["question_id"],
            "question": ex["question"],
            "answer": ex["answer"],
            "ground_truth_paragraphs": gt_ids,
            "semantic_results": sem_res,
            "graph_results": grp_res,
            "ppr_results": ppr_res,
            "semantic_metrics": {"p@5": sem_p5, "p@10": sem_p10, "r@5": sem_r5, "r@10": sem_r10, "mrr": sem_mrr},
            "graph_metrics": {"p@5": grp_p5, "p@10": grp_p10, "r@5": grp_r5, "r@10": grp_r10, "mrr": grp_mrr},
            "ppr_metrics": {"p@5": ppr_p5, "p@10": ppr_p10, "r@5": ppr_r5, "r@10": ppr_r10, "mrr": ppr_mrr},
        })

with open("evaluation_results.pkl", "wb") as f:
    pickle.dump(evaluation_results, f)
print(f"Saved evaluation results for {len(evaluation_results)} questions")

Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 50/50 [00:16<00:00,  3.10it/s]

Saved evaluation results for 50 questions


In [19]:
print(f"Evaluated on {NUM_EVAL} questions\n")
print("Method            | P@5    | P@10   | R@5    | R@10   | MRR")
print("-" * 65)

sem_avg = {k: np.mean(v) for k, v in semantic_metrics.items()}
grp_avg = {k: np.mean(v) for k, v in graph_metrics.items()}
ppr_avg = {k: np.mean(v) for k, v in ppr_metrics.items()}

print(f"Semantic Only     | {sem_avg['p@5']:.4f} | {sem_avg['p@10']:.4f} | {sem_avg['r@5']:.4f} | {sem_avg['r@10']:.4f} | {sem_avg['mrr']:.4f}")
print(f"Graph Enhanced    | {grp_avg['p@5']:.4f} | {grp_avg['p@10']:.4f} | {grp_avg['r@5']:.4f} | {grp_avg['r@10']:.4f} | {grp_avg['mrr']:.4f}")
print(f"PPR+IEF+DocLoc    | {ppr_avg['p@5']:.4f} | {ppr_avg['p@10']:.4f} | {ppr_avg['r@5']:.4f} | {ppr_avg['r@10']:.4f} | {ppr_avg['mrr']:.4f}")

Evaluated on 500 questions

Method            | P@5    | P@10   | R@5    | R@10   | MRR
-----------------------------------------------------------------
Semantic Only     | 0.2200 | 0.1200 | 0.5500 | 0.6000 | 0.7847
Graph Enhanced    | 0.2440 | 0.1440 | 0.6100 | 0.7200 | 0.8287
PPR+IEF+DocLoc    | 0.2280 | 0.1400 | 0.5700 | 0.7000 | 0.6797


# Error Analysis

In [20]:
with open("evaluation_results.pkl", "rb") as f:
    eval_results = pickle.load(f)

print(f"Loaded {len(eval_results)} evaluation results")

Loaded 50 evaluation results


In [ ]:
def find_bad_cases(results: list[dict], method: str = "ppr", threshold: float = 0.1) -> list[dict]:
    bad_cases = []
    for r in results:
        metrics = r[f"{method}_metrics"]
        if metrics["mrr"] < threshold or metrics["p@5"] < threshold:
            bad_cases.append({
                "question_id": r["question_id"],
                "question": r["question"],
                "answer": r["answer"],
                "ground_truth": r["ground_truth_paragraphs"],
                "retrieved": r[f"{method}_results"][:10],
                "metrics": metrics,
            })

    return bad_cases

ppr_bad = find_bad_cases(eval_results, method="ppr", threshold=0.1)
graph_bad = find_bad_cases(eval_results, method="graph", threshold=0.1)
semantic_bad = find_bad_cases(eval_results, method="semantic", threshold=0.1)

print(f"Bad cases (MRR < 0.1 or P@5 < 0.1):")
print(f"  PPR+IEF+DocLoc: {len(ppr_bad)}")
print(f"  Graph Enhanced: {len(graph_bad)}")
print(f"  Semantic Only: {len(semantic_bad)}")

Bad cases (MRR < 0.1 or P@5 < 0.1):
  PPR+IEF+DocLoc: 4
  Graph Enhanced: 3
  Semantic Only: 4


In [22]:
print("=" * 80)
print("WORST CASES - PPR+IEF+DocLoc")
print("=" * 80)

worst_ppr = sorted(ppr_bad, key=lambda x: x["metrics"]["mrr"])[:10]

for i, case in enumerate(worst_ppr, 1):
    print(f"\n--- Case {i} ---")
    print(f"Question ID: {case['question_id']}")
    print(f"Question: {case['question']}")
    print(f"Answer: {case['answer']}")
    print(f"Ground Truth Paragraphs: {case['ground_truth']}")
    print(f"Metrics: MRR={case['metrics']['mrr']:.4f}, P@5={case['metrics']['p@5']:.4f}, R@5={case['metrics']['r@5']:.4f}")
    print(f"\nTop 10 Retrieved:")
    for j, ret in enumerate(case['retrieved'], 1):
        marker = "✓" if ret["id"] in case["ground_truth"] else " "
        print(f"  {j}. [{marker}] {ret['id']} | {ret['text']}")

WORST CASES - PPR+IEF+DocLoc

--- Case 1 ---
Question ID: q_16
Question: Where was Tyler MacDuff's child educated?
Answer: Blair High School
Ground Truth Sentences: ['Dana MacDuff_0', 'Dana MacDuff_0']
Metrics: MRR=0.0000, P@5=0.0000, R@5=0.0000

Top 10 Retrieved:
  1. [ ] Ashland Community High School_1 | It was part of Maine School Administrative District 32, or MSAD 32, which serves Ashland, Garfield Plantation, Masardis, Oxbow, Portage Lake and Sheridan, Maine.
  2. [ ] Albert Tyler (biologist)_0 | Tyler was born in Brooklyn, New York.
  3. [ ] Edward VIII_3 | His mother was the eldest daughter of the Duke and Duchess of Teck (Francis and Mary Adelaide).
  4. [ ] Godfrey Baring_1 | His mother was Helen Graham, daughter of Sir James Graham, 2nd Baronet.
  5. [ ] Albert Tyler (biologist)_4 | Tyler completed his Ph.
  6. [ ] Ralph W. Tyler_3 | Tyler chaired the committee that developed the National Assessment of Educational Progress (NAEP).
  7. [ ] Ashland Community High School_3 | T

In [27]:
print("=" * 80)
print("WORST CASES - Graph Enhanced")
print("=" * 80)

worst_graph = sorted(graph_bad, key=lambda x: x["metrics"]["mrr"])[:10]

for i, case in enumerate(worst_graph, 1):
    print(f"\n--- Case {i} ---")
    print(f"Question ID: {case['question_id']}")
    print(f"Question: {case['question']}")
    print(f"Answer: {case['answer']}")
    print(f"Ground Truth Paragraphs: {case['ground_truth']}")
    print(f"Metrics: MRR={case['metrics']['mrr']:.4f}, P@5={case['metrics']['p@5']:.4f}, R@5={case['metrics']['r@5']:.4f}")
    print(f"\nTop 10 Retrieved:")
    for j, ret in enumerate(case['retrieved'], 1):
        marker = "✓" if ret["id"] in case["ground_truth"] else " "
        print(f"  {j}. [{marker}] {ret['id']} | {ret['text'][:80]}...")

WORST CASES - Graph Enhanced

--- Case 1 ---
Question ID: q_10
Question: Fast Cars, Danger, Fire and Knives includes guest appearances from which hip hop record executive?
Answer: Jaime Meline
Ground Truth Sentences: ['Fast Cars, Danger, Fire and Knives_2', 'El-P_0']
Metrics: MRR=0.0000, P@5=0.0000, R@5=0.0000

Top 10 Retrieved:
  1. [ ] Longterm Mentality_2 |  The album features guest appearances from Jhené Aiko, Schoolboy Q, Kendrick Lam...
  2. [ ] Animal Ambition_3 |  The album features guest appearances from Yo Gotti, Trey Songz, Kidd Kidd, Jada...
  3. [ ] Control Freek_3 |  It includes guest appearances from Tash's group Tha Alkaholiks in addition to g...
  4. [ ] Get Rich or Die Tryin' (soundtrack)_1 |  It features artists on the G-Unit label such as 50 Cent, Lloyd Banks, Young Buc...
  5. [ ] Fast Cars, Danger, Fire and Knives_0 | Fast Cars, Danger, Fire and Knives is an EP by American hip hop artist Aesop Roc...
  6. [ ] Lord Steppington_2 |  The record was produced entirely 

In [28]:
print("=" * 80)
print("CASES WHERE PPR PERFORMED WORSE THAN SEMANTIC")
print("=" * 80)

worse_than_semantic = []
for r in eval_results:
    if r["ppr_metrics"]["mrr"] < r["semantic_metrics"]["mrr"] - 0.1:
        worse_than_semantic.append({
            "question_id": r["question_id"],
            "question": r["question"],
            "answer": r["answer"],
            "ground_truth": r["ground_truth_sentences"],
            "semantic_mrr": r["semantic_metrics"]["mrr"],
            "ppr_mrr": r["ppr_metrics"]["mrr"],
            "semantic_results": r["semantic_results"][:5],
            "ppr_results": r["ppr_results"][:5],
        })

worse_than_semantic.sort(key=lambda x: x["semantic_mrr"] - x["ppr_mrr"], reverse=True)

print(f"Found {len(worse_than_semantic)} cases where PPR MRR is >0.1 worse than Semantic\n")

for i, case in enumerate(worse_than_semantic[:10], 1):
    print(f"\n--- Case {i} ---")
    print(f"Question ID: {case['question_id']}")
    print(f"Question: {case['question']}")
    print(f"Answer: {case['answer']}")
    print(f"Ground Truth: {case['ground_truth']}")
    print(f"Semantic MRR: {case['semantic_mrr']:.4f} | PPR MRR: {case['ppr_mrr']:.4f} (diff: {case['semantic_mrr'] - case['ppr_mrr']:.4f})")
    print(f"\nSemantic Top 5:")
    for j, ret in enumerate(case['semantic_results'], 1):
        marker = "✓" if ret["id"] in case["ground_truth"] else " "
        print(f"  {j}. [{marker}] {ret['id']} | {ret['text'][:70]}...")
    print(f"\nPPR Top 5:")
    for j, ret in enumerate(case['ppr_results'], 1):
        marker = "✓" if ret["id"] in case["ground_truth"] else " "
        print(f"  {j}. [{marker}] {ret['id']} | {ret['text'][:70]}...")

CASES WHERE PPR PERFORMED WORSE THAN SEMANTIC
Found 162 cases where PPR MRR is >0.1 worse than Semantic


--- Case 1 ---
Question ID: q_17
Question: Human Error" is the season finale of the third season of a tv show that aired on what network?
Answer: Fox
Ground Truth: ['Human Error (House)_0', 'House (TV series)_0']
Semantic MRR: 1.0000 | PPR MRR: 0.0000 (diff: 1.0000)

Semantic Top 5:
  1. [✓] Human Error (House)_0 | "Human Error" is the twenty-fourth episode and season finale of the th...
  2. [ ] List of Republic of Doyle episodes_5 |  The third season debuted on January 11, 2012 with 1,361,000 viewers, ...
  3. [ ] The Voice Portugal_6 |  In its third season, the show proved to be a hit and was subsequently...
  4. [ ] The Price of Free and Fair Elections_3 |  The season finale was originally supposed to be the 22nd episode, but...
  5. [ ] My Finale_1 |  It was originally broadcast as episodes 18 and 19 of season eight on ...

PPR Top 5:
  1. [ ] The Voice Portugal_6 |  In its th

In [29]:
print("=" * 80)
print("CASES WHERE PPR PERFORMED BETTER THAN SEMANTIC")
print("=" * 80)

better_than_semantic = []
for r in eval_results:
    if r["ppr_metrics"]["mrr"] > r["semantic_metrics"]["mrr"] + 0.1:
        better_than_semantic.append({
            "question_id": r["question_id"],
            "question": r["question"],
            "answer": r["answer"],
            "ground_truth": r["ground_truth_sentences"],
            "semantic_mrr": r["semantic_metrics"]["mrr"],
            "ppr_mrr": r["ppr_metrics"]["mrr"],
            "semantic_results": r["semantic_results"][:5],
            "ppr_results": r["ppr_results"][:5],
        })

better_than_semantic.sort(key=lambda x: x["ppr_mrr"] - x["semantic_mrr"], reverse=True)

print(f"Found {len(better_than_semantic)} cases where PPR MRR is >0.1 better than Semantic\n")

for i, case in enumerate(better_than_semantic[:10], 1):
    print(f"\n--- Case {i} ---")
    print(f"Question ID: {case['question_id']}")
    print(f"Question: {case['question']}")
    print(f"Answer: {case['answer']}")
    print(f"Ground Truth: {case['ground_truth']}")
    print(f"Semantic MRR: {case['semantic_mrr']:.4f} | PPR MRR: {case['ppr_mrr']:.4f} (diff: {case['ppr_mrr'] - case['semantic_mrr']:.4f})")
    print(f"\nSemantic Top 5:")
    for j, ret in enumerate(case['semantic_results'], 1):
        marker = "✓" if ret["id"] in case["ground_truth"] else " "
        print(f"  {j}. [{marker}] {ret['id']} | {ret['text'][:70]}...")
    print(f"\nPPR Top 5:")
    for j, ret in enumerate(case['ppr_results'], 1):
        marker = "✓" if ret["id"] in case["ground_truth"] else " "
        print(f"  {j}. [{marker}] {ret['id']} | {ret['text'][:70]}...")

CASES WHERE PPR PERFORMED BETTER THAN SEMANTIC
Found 86 cases where PPR MRR is >0.1 better than Semantic


--- Case 1 ---
Question ID: q_22
Question: Which band was founded first, Hole, the rock band that Courtney Love was a frontwoman of, or The Wolfhounds?
Answer: The Wolfhounds
Ground Truth: ['Courtney Love_0', 'Courtney Love_2', 'The Wolfhounds_0']
Semantic MRR: 0.1000 | PPR MRR: 1.0000 (diff: 0.9000)

Semantic Top 5:
  1. [ ] Florence Valentin_1 |  The band existed between 2002 and 2010....
  2. [ ] Antemasque (band)_1 |  Their first album featured Red Hot Chili Peppers's Flea on bass and D...
  3. [ ] Tír na nÓg (band)_3 |  In their early years, they toured the folk clubs of the United Kingdo...
  4. [ ] Blondie (band)_1 |  The band was a pioneer in the early American new wave and punk scenes...
  5. [ ] Throwing Muses_1 |  The group was originally fronted by two lead singers, Kristin Hersh, ...

PPR Top 5:
  1. [✓] The Wolfhounds_0 | The Wolfhounds are an indie pop/noise pop ban

In [1]:
import pandas as pd

error_analysis = []
for r in eval_results:
    error_analysis.append({
        "question_id": r["question_id"],
        "question": r["question"],
        "semantic_mrr": r["semantic_metrics"]["mrr"],
        "graph_mrr": r["graph_metrics"]["mrr"],
        "ppr_mrr": r["ppr_metrics"]["mrr"],
        "semantic_p5": r["semantic_metrics"]["p@5"],
        "graph_p5": r["graph_metrics"]["p@5"],
        "ppr_p5": r["ppr_metrics"]["p@5"],
        "num_gt_sentences": len(r["ground_truth_sentences"]),
    })

df = pd.DataFrame(error_analysis)
df.to_csv("error_analysis.csv", index=False)
print(f"Saved error analysis to error_analysis.csv")
print(f"\nSummary statistics:")
print(df[["semantic_mrr", "graph_mrr", "ppr_mrr", "semantic_p5", "graph_p5", "ppr_p5"]].describe())

NameError: name 'eval_results' is not defined

In [14]:
gt_doc_analysis = []
for r in ground_truth:
    gt_paragraphs = r["supporting_paragraphs"]
    docs = {pid.rsplit("__", 1)[0] for pid in gt_paragraphs}
    gt_doc_analysis.append({
        "question_id": r["question_id"],
        "num_paragraphs": len(gt_paragraphs),
        "num_docs": len(docs),
        "ratio": len(gt_paragraphs) / len(docs) if docs else 0,
    })

df_gt = pd.DataFrame(gt_doc_analysis)

print("Ground Truth Document Distribution")
print("=" * 50)
print(f"Total questions: {len(df_gt)}")
print(f"\nParagraphs per question: min={df_gt['num_paragraphs'].min()}, max={df_gt['num_paragraphs'].max()}, mean={df_gt['num_paragraphs'].mean():.2f}")
print(f"Documents per question: min={df_gt['num_docs'].min()}, max={df_gt['num_docs'].max()}, mean={df_gt['num_docs'].mean():.2f}")
print(f"Ratio (paragraphs/docs): mean={df_gt['ratio'].mean():.2f}")
print(f"\nDocument count distribution:")
print(df_gt["num_docs"].value_counts().sort_index())
print(f"\n% requiring 2+ documents: {100 * (df_gt['num_docs'] >= 2).mean():.1f}%")

Ground Truth Document Distribution
Total questions: 1000

Sentences per question: min=2, max=7, mean=2.38
Documents per question: min=2, max=2, mean=2.00
Ratio (sentences/docs): mean=1.19

Document count distribution:
num_docs
2    1000
Name: count, dtype: int64

% requiring 2+ documents: 100.0%


In [ ]:
driver.close()